In [486]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score, classification_report
from imblearn.over_sampling import SMOTE


In [487]:
df =pd.read_csv("/Users/preciousajilore/Documents/GitHub/torchmtlr/notebooks/prepostop.csv")
df.head()

,stxlocation,distal,penile,stxetiology,stxlength,#strictures,charlsons,cormorbidity,diabetes,copd,...,tissue_4.0,tissue_5.0,urethroplasty_1.0,urethroplasty_2.0,urethroplasty_3.0,urethroplasty_4.0,urethroplasty_5.0,urethroplasty_6.0,failure,time_to_event
0,0,0.0,1.0,2,3.5,1.0,0.0,1.0,1.0,0.0,...,False,False,False,True,False,False,False,False,0.0,5148
1,0,0.0,1.0,4,5,1.0,0.0,0.0,0.0,0.0,...,False,False,False,False,False,True,False,False,0.0,2563
2,2,0.0,0.0,1,3,1.0,0.0,0.0,0.0,0.0,...,False,False,True,False,False,False,False,False,0.0,7949
3,0,0.0,1.0,4,6.5,1.0,0.0,0.0,0.0,0.0,...,False,False,False,False,False,False,True,False,1.0,4282
4,1,0.0,0.0,1,1,1.0,1.0,1.0,0.0,0.0,...,False,False,True,False,False,False,False,False,0.0,7942


In [488]:
df.columns

Index(['stxlocation', 'distal', 'penile', 'stxetiology', 'stxlength',
       '#strictures', 'charlsons', 'cormorbidity', 'diabetes', 'copd',
       'smoker', 'bmi35+', 'prevprocedure', '#prevprocedures', 'cysto', 'open',
       'ordate', 'urine', 'stx_length_1', 'stx_length_2', 'stxetiology_0',
       'stxetiology_1', 'stxetiology_2', 'stxetiology_3', 'stxetiology_4',
       'stxetiology_5', 'stxetiology_6', 'stxlocation_0', 'stxlocation_1',
       'stxlocation_2', 'stxlocation_3', 'stxlocation_4', 'stxlocation_5',
       'stxlocation_6', 'stx_length_1.1', 'stx_length_2.1', 'cysto_0.0',
       'cysto_1.0', 'cysto_2.0', 'cysto_3.0', 'abx', 'erectilepre', 'uti',
       'los(days)', 'spc', 'tissue', 'transection', 'urethroplasty',
       'cathremoval', 'cathdays', 'tissue_0.0', 'tissue_1.0', 'tissue_2.0',
       'tissue_3.0', 'tissue_4.0', 'tissue_5.0', 'urethroplasty_1.0',
       'urethroplasty_2.0', 'urethroplasty_3.0', 'urethroplasty_4.0',
       'urethroplasty_5.0', 'urethroplasty_6.0

In [489]:
mask = df == "1, 5"
columns_with_value = mask.any(axis=0)
print("Columns containing '0,1':")
print(columns_with_value[columns_with_value].index.tolist())

Columns containing '0,1':
['stxlength']


In [490]:
df = df.drop(columns=["stxlocation","stxetiology","cysto","tissue","ordate","cathdays","cathremoval","stxlength"])

In [491]:
X = df.drop(columns = ["failure"])
y = df["failure"].astype(int)

In [492]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.5, random_state=42, stratify=y)

In [493]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

In [494]:
print(y.value_counts())


failure
0    1886
1     180
Name: count, dtype: int64


In [503]:
lr = LogisticRegression(class_weight = "balanced", max_iter=1000, random_state=42)
lr.fit(X_train_res, y_train_res)

rf = RandomForestClassifier(class_weight="balanced",n_estimators= 100, random_state=42)
rf.fit(X_train_res, y_train_res)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [498]:
metrics = {
    "accuracy": accuracy_score,
    "precision": precision_score,
    "recall": recall_score,
    "f1": f1_score,
    "roc_auc": roc_auc_score,
}


In [499]:
for name, model in [("LogisticRegression", lr), ("RandomForest", rf)]:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    print(f"\n{name} Metrics:")
    for metric_name, metric_func in metrics.items():
        if metric_name == "roc_auc":
            value = metric_func(y_test, y_proba)
        else:
            value = metric_func(y_test, y_pred)
        print(f"  {metric_name}: {value:.3f}")


LogisticRegression Metrics:
  accuracy: 0.898
  precision: 0.308
  recall: 0.133
  f1: 0.186
  roc_auc: 0.696

RandomForest Metrics:
  accuracy: 0.910
  precision: 0.333
  recall: 0.033
  f1: 0.061
  roc_auc: 0.720


In [500]:
coef = lr.coef_[0]
feature_importance = pd.Series(np.abs(coef), index=X.columns).sort_values(ascending=False)
print("\nTop Features (Logistic Regression):")
print(feature_importance.head(10))


Top Features (Logistic Regression):
stxlocation_3    2.836144
stxlocation_4    2.833699
stxetiology_4    2.723416
stxetiology_5    2.624457
stxetiology_3    2.619116
stxlocation_1    2.476654
cysto_2.0        2.403259
stxetiology_1    2.365590
stxlocation_0    2.271479
cysto_1.0        2.262660
dtype: float64


In [501]:
rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop Features (Random Forest):")
print(rf_importance.head(10))


Top Features (Random Forest):
time_to_event     0.089143
cormorbidity      0.073676
distal            0.056963
charlsons         0.055326
penile            0.053403
stxlocation_1     0.052959
open              0.047419
stx_length_1      0.042756
stx_length_1.1    0.040608
los(days)         0.035750
dtype: float64


In [504]:
# For LogisticRegression
coef_df = pd.DataFrame({'feature': X_train.columns, 'coef': lr.coef_[0]})
print(coef_df.sort_values(by='coef', ascending=False))

# For RandomForest
importances = rf.feature_importances_
importances_df = pd.DataFrame({'feature': X_train.columns, 'importance': importances})
print(importances_df.sort_values(by='importance', ascending=False))

              feature      coef
33          cysto_2.0  2.403259
32          cysto_1.0  2.262660
31          cysto_0.0  1.655689
52  urethroplasty_5.0  1.653313
43         tissue_1.0  1.435832
49  urethroplasty_2.0  1.387391
51  urethroplasty_4.0  1.380204
45         tissue_3.0  1.331143
44         tissue_2.0  1.182741
48  urethroplasty_1.0  1.129633
50  urethroplasty_3.0  0.855635
11               open  0.814045
14       stx_length_2  0.729750
30     stx_length_2.1  0.729750
0              distal  0.703971
42         tissue_0.0  0.603183
5            diabetes  0.486691
36        erectilepre  0.410130
4        cormorbidity  0.304331
53  urethroplasty_6.0  0.280255
39                spc  0.207550
8              bmi35+  0.104234
12              urine  0.103597
1              penile  0.081895
3           charlsons  0.079382
41      urethroplasty  0.015134
29     stx_length_1.1  0.014701
13       stx_length_1  0.014701
10    #prevprocedures  0.010904
54      time_to_event  0.000396
7       

In [ ]:
# Get top 10 feature names by importance
top_features = importances_df.sort_values('importance', ascending=False)['feature'].head(10).tolist()

# Now subset your data
X_train_top = X_train[top_features]
X_test_top = X_test[top_features]

# Fit your model again!
rf.fit(X_train_top, y_train)
y_pred = rf.predict(X_test_top)
print(classification_report(y_test, y_pred))




              precision    recall  f1-score   support

           0       0.92      0.98      0.95       943
           1       0.21      0.04      0.07        90

    accuracy                           0.90      1033
   macro avg       0.56      0.51      0.51      1033
weighted avg       0.85      0.90      0.87      1033



In [507]:
# Get top 10 feature names by importance
top_feat = coef_df.sort_values('coef', ascending=False)['feature'].head(10).tolist()

# Now subset your data
X_train_top = X_train[top_feat]
X_test_top = X_test[top_feat]

# Fit your model again!
lr.fit(X_train_top, y_train)
y_pred = lr.predict(X_test_top)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.80      0.86       943
           1       0.15      0.38      0.21        90

    accuracy                           0.76      1033
   macro avg       0.54      0.59      0.54      1033
weighted avg       0.86      0.76      0.80      1033

